# sanitise - live test on Colab (real Ollama + Presidio + spaCy)

Runs the Agent 1 code under real conditions on a free Colab GPU: real `llama3:8b`
through Ollama with JSON-schema-constrained decoding, real Presidio + spaCy
`en_core_web_lg`, the CLI and the HTTP API.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`, then run the cells top to bottom.

Note: this runs on Google's machines. Use only the bundled example / public
dataset fixtures here - the project's real deployment is meant to stay local.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 2. Get the code
Upload `3rd_sem_el_colab.zip` (made from the repo, without `.venv`) when prompted.
Once the repo is on GitHub you can `git clone` instead.

In [ ]:
import os
import zipfile

if not os.path.isdir("/content/3rd_sem_el"):
    from google.colab import files

    uploaded = files.upload()  # choose 3rd_sem_el_colab.zip
    zipfile.ZipFile(next(iter(uploaded))).extractall("/content")
    # Or, once pushed:  !git clone <repo-url> /content/3rd_sem_el

%cd /content/3rd_sem_el
!ls

## 3. Install Python dependencies (includes spaCy `en_core_web_lg`, ~400 MB)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps
import sys

sys.path.insert(
    0, "/content/3rd_sem_el/src"
)  # editable installs are only picked up after a kernel restart
!sanitise --version

## 4. Install and start Ollama

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os
import subprocess
import time
import urllib.request

ollama_log = open("/content/ollama.log", "w")
ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=ollama_log,
    stderr=subprocess.STDOUT,
    env={**os.environ, "OLLAMA_KEEP_ALIVE": "1h"},  # keep the model in GPU memory between calls
)
for _ in range(60):
    try:
        print(
            urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2).read().decode()
        )
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(open("/content/ollama.log").read()[-2000:])

## 5. Pull the models (llama3:8b primary, qwen2.5:7b-instruct fallback - ~4.7 GB each)

In [ ]:
!ollama pull llama3:8b > /dev/null 2>&1 && echo "llama3:8b ready"
!ollama pull qwen2.5:7b-instruct > /dev/null 2>&1 && echo "qwen2.5:7b-instruct ready"
!ollama list

## 6. Settings for the CLI / API in this session

In [ ]:
%env SANITISE_LLM_BACKEND=ollama
%env SANITISE_MODEL=llama3:8b
%env SANITISE_LLM_TIMEOUT_S=300
!sanitise health

## 7. Test suites

Fast suite (no model needed):

In [ ]:
!python -m pytest -q -p no:cacheprovider

Presidio + spaCy environment check (loads `en_core_web_lg`):

In [ ]:
!python -m pytest -q -p no:cacheprovider -m presidio

Live tests against the real model - constrained JSON call and full `detect()`, for both models:

In [ ]:
!python -m pytest -v -p no:cacheprovider -m ollama
!OLLAMA_TEST_MODEL=qwen2.5:7b-instruct python -m pytest -v -p no:cacheprovider -m ollama

## 8. The demo command (spec section 6) with the real model
The rule pass shows 0 spans until Person A replaces the `rule_detect` placeholder.

In [ ]:
!sanitise detect --input examples/case_demo.json --verbose
!sanitise detect --input examples/case_demo.json --verbose --model qwen2.5:7b-instruct

Batch mode, writing one result file per document (what `sanitise score` reads):

In [ ]:
!sanitise detect --input examples/case_demo.json --out /content/out
!ls /content/out && head -c 600 /content/out/*.json

## 9. HTTP API with the real model

In [ ]:
api_log = open("/content/api.log", "w")
api_proc = subprocess.Popen(
    ["sanitise", "serve", "--port", "8000"], stdout=api_log, stderr=subprocess.STDOUT
)
for _ in range(60):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:8000/v1/health", timeout=2).read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(open("/content/api.log").read()[-2000:])

In [ ]:
import json
import urllib.request


def post(path, body):
    req = urllib.request.Request(
        "http://127.0.0.1:8000" + path,
        data=json.dumps(body).encode(),
        headers={"content-type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=600) as resp:
            status, payload = resp.status, json.load(resp)
    except urllib.error.HTTPError as err:
        status, payload = err.code, json.load(err)
    print(f"POST {path} -> {status}")
    print(json.dumps(payload, indent=2)[:1500], end="\n\n")


text = (
    "My son is 32 years old and works as a lighthouse keeper near Port Isaac. "
    "He has had pelvic pain for ten weeks."
)
post("/v1/detect", {"text": text})
post("/v1/detect/rules", {"text": text})
post(
    "/v1/detect/llm",
    {
        "text": text,
        "explicit": [
            {"text": "My son", "category": "RELATION", "start": 0, "end": 6, "method": "rule"}
        ],
    },
)
post("/v1/detect", {"text": "x" * 2001})  # expect 413 DOC_TOO_LONG envelope

## 10. Model comparison (spec section 9: "benchmark both on five fixtures on Day 1")
Latency, retries needed and spans returned for each model. Point `FIXTURES` at
`data/fixtures` once Person D has added the corpus.

In [ ]:
from sanitise.agents.detector.pipeline import detect_with_trace
from sanitise.config import Settings
from sanitise.errors import PipelineError
from sanitise.llm import OllamaClient
from sanitise.loader import iter_documents

FIXTURES = "examples/case_demo.json"  # -> "data/fixtures"
docs = list(iter_documents(FIXTURES, skip_too_long=True))[:5]

for model in ["llama3:8b", "qwen2.5:7b-instruct"]:
    async with OllamaClient.from_settings(
        Settings(llm_backend="ollama", model=model, llm_timeout_s=300)
    ) as llm:
        for doc in docs:
            try:
                result, trace = await detect_with_trace(doc.doc_id, doc.text, llm)
                print(
                    f"{model:22} {doc.doc_id}  llm {result.meta.llm_ms:6d} ms  "
                    f"retries {result.meta.retries}  quasi spans {result.meta.llm_span_count}"
                )
                for span in trace.llm_spans:
                    print(f"{'':24}- {span.text}  ({span.rationale})")
            except PipelineError as exc:
                print(f"{model:22} {doc.doc_id}  FAILED {exc}")

## 11. Clean up

In [ ]:
api_proc.terminate()
ollama_proc.terminate()